# I. Initialization & Environment Setup

In [1]:
import time

_nb_start_time = time.perf_counter()

In [2]:
# 1. Setup & Path
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from pathlib import Path

from core.settings import TradingConfig
from core.contracts import EngineInput, FilterPack
from core.utils import SystemUtils as SU

# Vertical slice imports
from data_pipeline import generate_features
from walk_forward import AlphaEngine, create_walk_forward_analyzer, run_headless_simulation

config = TradingConfig()

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output



In [3]:
# 2. Path Settings
PROCESSED_DATA_DIR = Path(
    r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data\processed"
)
OUTPUT_DIR = Path(
    r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output"
)
DATA_DIR = Path(r"C:\Users\ping\Files_win10\python\py311\stocks\data")
DATA_PATH_OHLCV = DATA_DIR / "df_OHLCV_stocks_etfs.parquet"
DATA_PATH_INDICES = DATA_DIR / "df_indices.parquet"

print(f"PROCESSED_DATA_DIR: {PROCESSED_DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

PROCESSED_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data\processed
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output
DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
DATA_PATH_OHLCV: C:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: C:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


# II. Data Pipeline
Choose to either reload features quickly from Parquet cache or run the ~6-minute feature generation process.

### Option A: Reload Existing Features from Parquet (Quick)

In [4]:
features_df = pd.read_parquet(PROCESSED_DATA_DIR / "features_df.parquet")
macro_df = pd.read_parquet(PROCESSED_DATA_DIR / "macro_df.parquet")
df_close_wide = pd.read_parquet(PROCESSED_DATA_DIR / "df_close_wide.parquet")
df_atrp_wide = pd.read_parquet(PROCESSED_DATA_DIR / "df_atrp_wide.parquet")
df_trp_wide = pd.read_parquet(PROCESSED_DATA_DIR / "df_trp_wide.parquet")

df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
df_fed = pd.read_parquet(
    DATA_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet",
    engine="pyarrow",
)
print("✅ Cached Data Reloaded.")

✅ Cached Data Reloaded.


### Option B: Re-generate All Features (Slow)

In [5]:
# print("⏳ Loading DataFrames... takes ~6 minutes")
# df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
# df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
# df_fed = pd.read_parquet(
#     DATA_DIR / "High_Yield_Spread_T10Y2Y_Spread.parquet", engine="pyarrow"
# )

# print("⚡ Generating Features... takes about 6 minutes")
# features_df, macro_df = generate_features(
#     df_ohlcv=df_ohlcv,
#     df_indices=df_indices,
#     df_fed=df_fed,
#     config=config,
# )

# print("🚀 Unstacking Wide Matrices...")
# df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
# df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)
# df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# if config.handle_zeros_as_nan:
#     df_close_wide = df_close_wide.replace(0, np.nan)

# df_close_wide = df_close_wide.ffill(limit=config.max_data_gap_ffill)
# df_close_wide = df_close_wide.fillna(config.nan_price_replacement)

# print(
#     f"✅ Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
# )

In [6]:
# # Optional: Save the freshly generated datasets to the correct processed directory
# PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# features_df.to_parquet(PROCESSED_DATA_DIR / "features_df.parquet")
# macro_df.to_parquet(PROCESSED_DATA_DIR / "macro_df.parquet")
# df_close_wide.to_parquet(PROCESSED_DATA_DIR / "df_close_wide.parquet")
# df_atrp_wide.to_parquet(PROCESSED_DATA_DIR / "df_atrp_wide.parquet")
# df_trp_wide.to_parquet(PROCESSED_DATA_DIR / "df_trp_wide.parquet")

# print("💾 Processed Data Cached Successfully.")

# III. Walk-Forward Performance Engine

In [7]:
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("🎯 Master AlphaEngine Ready.")

🎯 Master AlphaEngine Ready.


In [ ]:
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-05-01"),
    lookback_period=10,
    holding_period=5,
    metric="Sharpe (TRP)",
    benchmark_ticker=config.benchmark_ticker,
    rank_start=1,
    rank_end=200,
    debug=False,
)

analyzer1, stage1_pack = create_walk_forward_analyzer(
    engine, _inputs, universe_subset=None
)
analyzer1.show()

In [9]:
result = SU.map_analyzer(analyzer=analyzer1)
SU.list_analyzer_paths(result)

[SEARCH] ANALYZER SIMULATION MAP
[  0] [DATA] audit_pack (EngineOutput)
[  1]   [PLOT] portfolio_series (shape=(17,))
[  2]   [PLOT] benchmark_series (shape=(17,))
[  3]   [CALC] normalized_plot_data (shape=(17, 200))
[  4]   [FILE] tickers (len=200)
[  5]     [DOC] index_0 (str)
[  6]     [DOC] index_1 (str)
[  7]     [DOC] index_2 (str)
[  8]     [DOC] index_3 (str)
[  9]     [DOC] index_4 (str)
[ 10]     [DOC] index_5 (str)
[ 11]     [DOC] index_6 (str)
[ 12]     [DOC] index_7 (str)
[ 13]     [DOC] index_8 (str)
[ 14]     [DOC] index_9 (str)
[ 15]     [DOC] index_10 (str)
[ 16]     [DOC] index_11 (str)
[ 17]     [DOC] index_12 (str)
[ 18]     [DOC] index_13 (str)
[ 19]     [DOC] index_14 (str)
[ 20]     [DOC] index_15 (str)
[ 21]     [DOC] index_16 (str)
[ 22]     [DOC] index_17 (str)
[ 23]     [DOC] index_18 (str)
[ 24]     [DOC] index_19 (str)
[ 25]     [DOC] index_20 (str)
[ 26]     [DOC] index_21 (str)
[ 27]     [DOC] index_22 (str)
[ 28]     [DOC] index_23 (str)
[ 29]     [DOC]

In [10]:
# fetch by id
print(f"fetch by id=1:\not{SU.fetch(1, result)}\n")

# fetch by name
print(f'fetch by name = "portfolio_series":\n{SU.fetch("portfolio_series", result)}')

 [INFO] INDEX: [1]
 [LABEL] NAME:  portfolio_series
 [FILE] PATH:  audit_pack -> portfolio_series



Date
2026-04-17    1.000000
2026-04-20    1.010604
2026-04-21    1.016630
2026-04-22    1.030356
2026-04-23    1.039983
2026-04-24    1.052968
2026-04-27    1.053197
2026-04-28    1.048946
2026-04-29    1.069061
2026-04-30    1.092390
2026-05-01    1.103319
2026-05-04    1.105369
2026-05-05    1.125884
2026-05-06    1.133777
2026-05-07    1.119758
2026-05-08    1.137621
2026-05-11    1.154806
dtype: float64

fetch by id=1:
otDate
2026-04-17    1.000000
2026-04-20    1.010604
2026-04-21    1.016630
2026-04-22    1.030356
2026-04-23    1.039983
2026-04-24    1.052968
2026-04-27    1.053197
2026-04-28    1.048946
2026-04-29    1.069061
2026-04-30    1.092390
2026-05-01    1.103319
2026-05-04    1.105369
2026-05-05    1.125884
2026-05-06    1.133777
2026-05-07    1.119758
2026-05-08    1.137621
2026-05-11    1.154806
dtype: float64

 [INFO] INDEX: [1]
 [LABEL] NAME:  portfolio_series
 [FILE] PATH:  audit_pack -> portfolio_series



Date
2026-04-17    1.000000
2026-04-20    1.010604
2026-04-21    1.016630
2026-04-22    1.030356
2026-04-23    1.039983
2026-04-24    1.052968
2026-04-27    1.053197
2026-04-28    1.048946
2026-04-29    1.069061
2026-04-30    1.092390
2026-05-01    1.103319
2026-05-04    1.105369
2026-05-05    1.125884
2026-05-06    1.133777
2026-05-07    1.119758
2026-05-08    1.137621
2026-05-11    1.154806
dtype: float64

fetch by name = "portfolio_series":
Date
2026-04-17    1.000000
2026-04-20    1.010604
2026-04-21    1.016630
2026-04-22    1.030356
2026-04-23    1.039983
2026-04-24    1.052968
2026-04-27    1.053197
2026-04-28    1.048946
2026-04-29    1.069061
2026-04-30    1.092390
2026-05-01    1.103319
2026-05-04    1.105369
2026-05-05    1.125884
2026-05-06    1.133777
2026-05-07    1.119758
2026-05-08    1.137621
2026-05-11    1.154806
dtype: float64


# IV. Audit: Walk-Forward Analyzer Verification
Checking execution dates and verifying calculated drift weight statistics against baseline simulation audit packs.

In [11]:
start_date = SU.fetch("start_date", result)
decision_date = SU.fetch("decision_date", result)
buy_date = SU.fetch("buy_date", result)
holding_end_date = SU.fetch("holding_end_date", result)

print(f"Start Date:       {start_date}")
print(f"Decision Date:    {decision_date}")
print(f"Buy Date:         {buy_date}")
print(f"Holding End Date: {holding_end_date}")

target_tickers = SU.fetch("tickers", result)
print(f"target_tickers: {target_tickers}")

 [INFO] INDEX: [232]
 [LABEL] NAME:  start_date
 [FILE] PATH:  audit_pack -> start_date



Timestamp('2026-04-17 00:00:00')

 [INFO] INDEX: [233]
 [LABEL] NAME:  decision_date
 [FILE] PATH:  audit_pack -> decision_date



Timestamp('2026-05-01 00:00:00')

 [INFO] INDEX: [234]
 [LABEL] NAME:  buy_date
 [FILE] PATH:  audit_pack -> buy_date



...

 [INFO] INDEX: [235]
 [LABEL] NAME:  holding_end_date
 [FILE] PATH:  audit_pack -> holding_end_date



Timestamp('2026-05-11 00:00:00')

Start Date:       2026-04-17 00:00:00
Decision Date:    2026-05-01 00:00:00
Buy Date:         2026-05-04 00:00:00
Holding End Date: 2026-05-11 00:00:00
 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'CNC',
 'CVE',
 'BIL',
 'MINT',
 'NXPI',
 'VBIL',
 'PWR',
 'MO',
 'USFR',
 'OVV',
 'INTC',
 'FANG',
 'STM',
 'BOXX',
 'MOH',
 'MUSA',
 'PR',
 'DINO',
 'TWLO',
 'TXN',
 'ADM',
 'DVN',
 'GNRC',
 'NUE',
 'SU',
 'MPC',
 'ON',
 'QCOM',
 'URI',
 'USO',
 'GOOG',
 'AMD',
 'BKR',
 'CASY',
 'STX',
 'UNH',
 'ELV',
 'GOOGL',
 'MCHP',
 'BE',
 'TRP',
 'TEAM',
 'PULS',
 'SWKS',
 'PSX',
 'QRVO',
 'NOK',
 'ZM',
 'NVT',
 'SOXL',
 'STLD',
 'SOXX',
 'CAT',
 'HAL',
 'CNQ',
 'SNDK',
 'APA',
 'SMH',
 'TRGP',
 'EOG',
 'EWT',
 'XLE',
 'VLO',
 'LYB',
 'OKE',
 'ET',
 'JEPQ',
 'MTCH',
 'OXY',
 'TROW',
 'ARM',
 'MTZ',
 'EME',
 'CBOE',
 'QQQI',
 'KDP',
 'MDLZ',
 'PBR',
 'MU',
 'FLEX',
 'EPD',
 'FNDX',
 'MRVL',
 'LNG',
 'QQQM',
 'HUM',
 'QQQ',
 'IYW',
 'QLD',
 'TQQQ',
 'STRL',
 'WMB',
 'JPST',
 'TTMI',
 'SPMO',
 'NTR',
 'DOW',
 'CARR',
 'CVS',
 'CSCO',
 'XLK',
 'ASX',
 'FIX',
 'TECL',
 'FTI',
 'WCC',
 'SWK',
 'ADP',
 'FLR',
 'CP',
 'AVB',
 'WST',
 'SLB',
 'JAAA',
 'NTAP',
 'UNP',
 'IRM',
 'INVH',
 

target_tickers: ['SGOV', 'SHV', 'CNC', 'CVE', 'BIL', 'MINT', 'NXPI', 'VBIL', 'PWR', 'MO', 'USFR', 'OVV', 'INTC', 'FANG', 'STM', 'BOXX', 'MOH', 'MUSA', 'PR', 'DINO', 'TWLO', 'TXN', 'ADM', 'DVN', 'GNRC', 'NUE', 'SU', 'MPC', 'ON', 'QCOM', 'URI', 'USO', 'GOOG', 'AMD', 'BKR', 'CASY', 'STX', 'UNH', 'ELV', 'GOOGL', 'MCHP', 'BE', 'TRP', 'TEAM', 'PULS', 'SWKS', 'PSX', 'QRVO', 'NOK', 'ZM', 'NVT', 'SOXL', 'STLD', 'SOXX', 'CAT', 'HAL', 'CNQ', 'SNDK', 'APA', 'SMH', 'TRGP', 'EOG', 'EWT', 'XLE', 'VLO', 'LYB', 'OKE', 'ET', 'JEPQ', 'MTCH', 'OXY', 'TROW', 'ARM', 'MTZ', 'EME', 'CBOE', 'QQQI', 'KDP', 'MDLZ', 'PBR', 'MU', 'FLEX', 'EPD', 'FNDX', 'MRVL', 'LNG', 'QQQM', 'HUM', 'QQQ', 'IYW', 'QLD', 'TQQQ', 'STRL', 'WMB', 'JPST', 'TTMI', 'SPMO', 'NTR', 'DOW', 'CARR', 'CVS', 'CSCO', 'XLK', 'ASX', 'FIX', 'TECL', 'FTI', 'WCC', 'SWK', 'ADP', 'FLR', 'CP', 'AVB', 'WST', 'SLB', 'JAAA', 'NTAP', 'UNP', 'IRM', 'INVH', 'NVO', 'ENB', 'WDC', 'AMZN', 'SCHD', 'FSLR', 'TEVA', 'AKAM', 'CF', 'SNPS', 'SBUX', 'MTUM', 'CDNS', 'VGT'

In [12]:
def get_value_by_key(search_term, data):
    """
    Search for a value by its name or its full path string.
    """
    for item in data:
        # Check if the search term matches the name OR the full path
        if item.get("name") == search_term or item.get("path") == search_term:
            return item.get("obj")

    return f"Error: Key '{search_term}' not found in the dataset."

In [13]:
raw_prices = get_value_by_key(
    "audit_pack -> debug_data -> portfolio_raw_components -> prices", result
)
raw_prices

Ticker,SGOV,SHV,CNC,CVE,BIL,MINT,NXPI,VBIL,PWR,MO,...,APD,BIIB,SCHB,IEX,SCHX,ITOT,SPXL,SSO,AAPL,IVE
Date,,,,,,,,,,,,,,,,,,,,,
2026-04-17,99.9645,109.623,38.17,24.51,91.0124,99.8871,216.03,75.1372,601.88,64.17,...,291.81,177.35,27.43,205.189,27.99,155.59,238.73,61.62,269.981,220.89
2026-04-20,99.9745,109.623,38.31,24.91,91.0223,99.8971,221.34,75.1471,604.97,64.61,...,296.15,183.34,27.40,205.627,27.96,155.49,237.33,61.37,272.799,221.12
2026-04-21,99.9944,109.633,39.14,25.60,91.0323,99.9070,224.50,75.1571,605.89,64.65,...,294.78,185.95,27.23,204.063,27.78,154.45,232.68,60.58,265.925,219.76
2026-04-22,100.0040,109.652,38.93,25.92,91.0323,99.9169,225.75,75.1670,613.78,65.18,...,296.76,190.07,27.48,202.907,28.04,155.91,239.69,61.78,272.919,220.16
2026-04-23,100.0140,109.662,41.09,26.45,91.0521,99.9666,241.16,75.1769,633.44,67.15,...,303.65,187.88,27.38,205.906,27.94,155.24,236.62,61.31,273.178,220.85
2026-04-24,100.0440,109.692,41.82,26.29,91.0720,99.9865,244.04,75.1968,624.84,66.88,...,301.76,184.38,27.56,204.013,28.14,156.34,242.14,62.21,270.810,220.63
2026-04-27,100.0440,109.702,43.50,26.78,91.0820,100.0060,236.87,75.1968,637.28,66.00,...,302.38,180.67,27.61,206.654,28.17,156.55,243.30,62.42,267.364,220.06
2026-04-28,100.0640,109.712,49.57,27.42,91.0919,100.0260,230.39,75.2068,630.94,67.80,...,303.35,183.38,27.46,204.631,28.03,155.72,239.71,61.80,270.461,220.20
2026-04-29,100.0740,109.712,53.98,28.75,91.0919,100.0560,289.25,75.2167,628.60,68.20,...,302.50,194.38,27.44,216.568,28.02,155.62,239.60,61.79,269.921,220.91


In [14]:
# raw_prices = SU.fetch("full_universe_ranking", result)
raw_prices = SU.fetch(277, result)
# raw_prices = SU.fetch("prices", result)

 [INFO] INDEX: [277]
 [LABEL] NAME:  prices
 [FILE] PATH:  audit_pack -> debug_data -> portfolio_raw_components -> prices



Ticker,SGOV,SHV,CNC,CVE,BIL,MINT,NXPI,VBIL,PWR,MO,...,APD,BIIB,SCHB,IEX,SCHX,ITOT,SPXL,SSO,AAPL,IVE
Date,,,,,,,,,,,,,,,,,,,,,
2026-04-17,99.9645,109.623,38.17,24.51,91.0124,99.8871,216.03,75.1372,601.88,64.17,...,291.81,177.35,27.43,205.189,27.99,155.59,238.73,61.62,269.981,220.89
2026-04-20,99.9745,109.623,38.31,24.91,91.0223,99.8971,221.34,75.1471,604.97,64.61,...,296.15,183.34,27.40,205.627,27.96,155.49,237.33,61.37,272.799,221.12
2026-04-21,99.9944,109.633,39.14,25.60,91.0323,99.9070,224.50,75.1571,605.89,64.65,...,294.78,185.95,27.23,204.063,27.78,154.45,232.68,60.58,265.925,219.76
2026-04-22,100.0040,109.652,38.93,25.92,91.0323,99.9169,225.75,75.1670,613.78,65.18,...,296.76,190.07,27.48,202.907,28.04,155.91,239.69,61.78,272.919,220.16
2026-04-23,100.0140,109.662,41.09,26.45,91.0521,99.9666,241.16,75.1769,633.44,67.15,...,303.65,187.88,27.38,205.906,27.94,155.24,236.62,61.31,273.178,220.85
2026-04-24,100.0440,109.692,41.82,26.29,91.0720,99.9865,244.04,75.1968,624.84,66.88,...,301.76,184.38,27.56,204.013,28.14,156.34,242.14,62.21,270.810,220.63
2026-04-27,100.0440,109.702,43.50,26.78,91.0820,100.0060,236.87,75.1968,637.28,66.00,...,302.38,180.67,27.61,206.654,28.17,156.55,243.30,62.42,267.364,220.06
2026-04-28,100.0640,109.712,49.57,27.42,91.0919,100.0260,230.39,75.2068,630.94,67.80,...,303.35,183.38,27.46,204.631,28.03,155.72,239.71,61.80,270.461,220.20
2026-04-29,100.0740,109.712,53.98,28.75,91.0919,100.0560,289.25,75.2167,628.60,68.20,...,302.50,194.38,27.44,216.568,28.02,155.62,239.60,61.79,269.921,220.91


In [15]:
from typing import cast

# --- 1. Extract Drift Data ---
raw_prices = get_value_by_key(
    "audit_pack -> debug_data -> portfolio_raw_components -> prices", result
)
raw_atrp = get_value_by_key(
    "audit_pack -> debug_data -> portfolio_raw_components -> atrp", result
)
raw_trp = get_value_by_key(
    "audit_pack -> debug_data -> portfolio_raw_components -> trp", result
)

metric_suffixes = ["gain", "sharpe", "sharpe_atrp", "sharpe_trp"]
period_prefixes = ["full", "lookback", "holding"]
audit_data = {
    f"{p}_p_{s}": get_value_by_key(f"{p}_p_{s}", result)
    for s in metric_suffixes
    for p in period_prefixes
}

metric_types = ["Log Gain", "Sharpe", "Sharpe (ATRP)", "Sharpe (TRP)"]
period_names = ["Full", "Lookback", "Holding"]
period_slices = {
    "Full": (start_date, holding_end_date),
    "Lookback": (start_date, decision_date),
    "Holding": (buy_date, holding_end_date),
}

portfolio_results = {}

# --- 2. Calculate Drift Weights & Expected Returns ---
for name, (s_date, e_date) in period_slices.items():

    raw_prices = cast(pd.DataFrame, raw_prices)
    raw_atrp = cast(pd.DataFrame, raw_atrp)
    raw_trp = cast(pd.DataFrame, raw_trp)

    p_slice = raw_prices.loc[s_date:e_date]
    a_slice = raw_atrp.loc[s_date:e_date]
    t_slice = raw_trp.loc[s_date:e_date]

    norm_prices = p_slice / p_slice.iloc[0]
    weights = norm_prices.div(norm_prices.sum(axis=1), axis=0)

    equity_curve = norm_prices.mean(axis=1)
    returns = equity_curve.pct_change().dropna()

    port_atrp = (weights * a_slice).sum(axis=1).loc[returns.index]
    port_trp = (weights * t_slice).sum(axis=1).loc[returns.index]

    portfolio_results[f"{name} Log Gain"] = np.log(equity_curve.iloc[-1])
    portfolio_results[f"{name} Sharpe"] = (returns.mean() / returns.std()) * np.sqrt(
        252
    )
    portfolio_results[f"{name} Sharpe (ATRP)"] = returns.mean() / port_atrp.mean()
    portfolio_results[f"{name} Sharpe (TRP)"] = returns.mean() / port_trp.mean()

# --- 3. Evaluate Discrepancy ---
print(f"{'METRIC':<25} | {'CALCULATED':<12} | {'AUDIT PACK':<12} | {'MATCH?'}")
for m_type in metric_types:
    print("-" * 75)
    for p_name in period_names:
        key = f"{p_name} {m_type}"
        sfx = m_type.lower().replace(" ", "_").replace("(", "").replace(")", "")
        audit_key = f"{p_name.lower()}_p_{'gain' if sfx == 'log_gain' else sfx}"

        calc_val, audit_val = portfolio_results[key], audit_data[audit_key]
        is_match = np.isclose(float(calc_val), float(audit_val or 0), rtol=1e-05)
        print(
            f"{key:<25} | {calc_val:<12.6f} | {audit_val:<12.6f} | {'✅ YES' if is_match else '❌ NO'}"
        )

for key, val in portfolio_results.items():
    p, m = key.split(" ", 1)
    sfx = m.lower().replace(" ", "_").replace("(", "").replace(")", "")
    audit_key = f"{p.lower()}_p_{'gain' if sfx == 'log_gain' else sfx}"
    assert np.isclose(
        float(val or 0.0), float(audit_data[audit_key] or 0.0), rtol=1e-05
    )

print("Verification complete. Drift weights match the holding period metrics.")

METRIC                    | CALCULATED   | AUDIT PACK   | MATCH?
---------------------------------------------------------------------------
Full Log Gain             | 0.143932     | 0.143932     | ✅ YES
Lookback Log Gain         | 0.098323     | 0.098323     | ✅ YES
Holding Log Gain          | 0.041702     | 0.041702     | ✅ YES
---------------------------------------------------------------------------
Full Sharpe               | 15.789618    | 15.789618    | ✅ YES
Lookback Sharpe           | 20.081569    | 20.081569    | ✅ YES
Holding Sharpe            | 11.107559    | 11.107559    | ✅ YES
---------------------------------------------------------------------------
Full Sharpe (ATRP)        | 0.290284     | 0.290284     | ✅ YES
Lookback Sharpe (ATRP)    | 0.320815     | 0.320815     | ✅ YES
Holding Sharpe (ATRP)     | 0.270137     | 0.270137     | ✅ YES
---------------------------------------------------------------------------
Full Sharpe (TRP)         | 0.258178     | 0.258178    

# V. Audit: Individual Feature Math
Step-by-step verification comparing pipeline features against manual formulations for a chosen target ticker.

### Group A: Basic Features (Returns, Momentum, Drawdown, Consistency, TRP)

In [26]:
# verification_ticker = "NVDA"
# target_date = pd.Timestamp("2026-04-24")
# lookback_days = 120

verification_ticker = "A"
target_date = pd.Timestamp("2026-05-01")
lookback_days = 10


ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")

if target_date not in ticker_prices.index:
    new_target = ticker_prices.index[ticker_prices.index <= target_date][-1]
    print(f"Adjusting target date from {target_date.date()} to {new_target.date()}")
    target_date = new_target

raw_slice = ticker_prices.loc[:target_date].tail(90).copy()
feat_row = ticker_features.loc[target_date]

manual_ret_1d = raw_slice["Adj Close"].pct_change().iloc[-1]
manual_mom_21 = raw_slice["Adj Close"].pct_change(21).iloc[-1]

rolling_max_21 = raw_slice["Adj Close"].rolling(window=21).max()
manual_dd_21 = (raw_slice["Adj Close"] / rolling_max_21 - 1).iloc[-1]

daily_rets = raw_slice["Adj Close"].pct_change()
manual_consist = (daily_rets > 0).rolling(window=5).mean().iloc[-1]

prev_close = raw_slice["Adj Close"].shift(1)
tr_val = pd.concat(
    [
        (raw_slice["Adj High"] - raw_slice["Adj Low"]),
        (raw_slice["Adj High"] - prev_close).abs(),
        (raw_slice["Adj Low"] - prev_close).abs(),
    ],
    axis=1,
).max(axis=1)
manual_trp = (tr_val / raw_slice["Adj Close"]).iloc[-1]

easy_verification = pd.DataFrame(
    {
        "Feature": ["Ret_1d", "Mom_21", "DD_21", "Consistency", "TRP"],
        "Manual_Value": [
            manual_ret_1d,
            manual_mom_21,
            manual_dd_21,
            manual_consist,
            manual_trp,
        ],
        "Feature_DF_Value": [
            feat_row["Ret_1d"],
            feat_row["Mom_21"],
            feat_row["DD_21"],
            feat_row["Consistency"],
            feat_row["TRP"],
        ],
    }
)
easy_verification["Abs_Error"] = (
    easy_verification["Manual_Value"] - easy_verification["Feature_DF_Value"]
).abs()

print(f"\nVerification for {verification_ticker} on {target_date.date()}")
print(
    easy_verification.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:.6f}".format,
            "Feature_DF_Value": "{:.6f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

if easy_verification["Abs_Error"].max() < 1e-10:
    print("\n✅ Easy Group Verified.")
else:
    print("\n❌ Discrepancy detected.")


Verification for A on 2026-05-01
    Feature Manual_Value Feature_DF_Value Abs_Error
     Ret_1d    -0.008914        -0.008914  0.00e+00
     Mom_21    -0.000175        -0.000175  0.00e+00
      DD_21    -0.062003        -0.062003  0.00e+00
Consistency     0.400000         0.400000  0.00e+00
        TRP     0.021306         0.021306  0.00e+00

✅ Easy Group Verified.


### Group B: Smoothed Indicators (RSI, ATRP, Autocorrelation)

In [27]:
ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")
feat_row = ticker_features.loc[target_date]

adj_close = ticker_prices["Adj Close"]
adj_high = ticker_prices["Adj High"]
adj_low = ticker_prices["Adj Low"]
rsi_period = config.rsi_period
atr_period = config.atr_period

# 1. Wilder's RSI
delta = adj_close.diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.ewm(alpha=1 / rsi_period, adjust=False).mean()
avg_loss = loss.ewm(alpha=1 / rsi_period, adjust=False).mean()
rs = avg_gain / avg_loss
manual_rsi = (100 - (100 / (1 + rs))).loc[target_date]

# 2. ATRP
prev_close = adj_close.shift(1)
tr = pd.concat(
    [(adj_high - adj_low), (adj_high - prev_close).abs(), (adj_low - prev_close).abs()],
    axis=1,
).max(axis=1)
manual_atr = tr.ewm(alpha=1 / atr_period, adjust=False).mean()
manual_atrp = (manual_atr / adj_close).loc[target_date]

# 3. Lag-1 Correlation
rets = adj_close.pct_change()
manual_autocorr = rets.rolling(window=15).corr(rets.shift(1)).loc[target_date]

group2_verification = pd.DataFrame(
    {
        "Feature": ["RSI", "ATRP", "AutoCorr_15"],
        "Manual_Value": [manual_rsi, manual_atrp, manual_autocorr],
        "Feature_DF_Value": [
            feat_row["RSI"],
            feat_row["ATRP"],
            feat_row["AutoCorr_15"],
        ],
    }
)
group2_verification["Abs_Error"] = (
    group2_verification["Manual_Value"] - group2_verification["Feature_DF_Value"]
).abs()

print(f"Verification for {verification_ticker} on {target_date.date()}")
print(
    group2_verification.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:.6f}".format,
            "Feature_DF_Value": "{:.6f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

if group2_verification["Abs_Error"].max() < 1e-8:
    print("\n✅ Smoothed Group Verified.")
else:
    print("\n❌ Discrepancy detected.")

Verification for A on 2026-05-01
    Feature Manual_Value Feature_DF_Value Abs_Error
        RSI    46.042348        46.042348  0.00e+00
       ATRP     0.030175         0.030175  0.00e+00
AutoCorr_15    -0.205047        -0.205047  0.00e+00

✅ Smoothed Group Verified.


### Group C: Quality & Acceleration Features (Staleness, Dollar Volume, Convexity)

In [28]:
q_window = config.quality_window
q_min = config.quality_min_periods

ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")
feat_row = ticker_features.loc[target_date]

is_stale = (
    (ticker_prices["Volume"] == 0)
    | (ticker_prices["Adj High"] == ticker_prices["Adj Low"])
).astype(int)
manual_stale_pct = (
    is_stale.rolling(window=q_window, min_periods=q_min).mean().loc[target_date]
)

has_same_vol = (ticker_prices["Volume"].diff() == 0).astype(int)
manual_same_vol = (
    has_same_vol.rolling(window=q_window, min_periods=q_min).sum().loc[target_date]
)

dollar_vol = ticker_prices["Adj Close"] * ticker_prices["Volume"]
manual_med_dvol = (
    dollar_vol.rolling(window=q_window, min_periods=q_min).median().loc[target_date]
)
s_idx = ticker_features.index.get_indexer_for(pd.Index([target_date]))[0]
slope_t = ticker_features["Slope_P_5"].iloc[s_idx]
slope_t_minus_2 = ticker_features["Slope_P_5"].iloc[s_idx - 2]
manual_convexity = slope_t - slope_t_minus_2

final_verification = pd.DataFrame(
    {
        "Feature": [
            "RollingStalePct",
            "RollingSameVolCount",
            "RollMedDollarVol",
            "Convexity",
        ],
        "Manual_Value": [
            manual_stale_pct,
            manual_same_vol,
            manual_med_dvol,
            manual_convexity,
        ],
        "Feature_DF_Value": [
            feat_row["RollingStalePct"],
            feat_row["RollingSameVolCount"],
            feat_row["RollMedDollarVol"],
            feat_row["Convexity"],
        ],
    }
)
final_verification["Abs_Error"] = (
    final_verification["Manual_Value"] - final_verification["Feature_DF_Value"]
).abs()

print(f"--- Final Group Verification: {verification_ticker} ({target_date.date()}) ---")
print(
    final_verification.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:,.4f}".format,
            "Feature_DF_Value": "{:,.4f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

if final_verification["Abs_Error"].max() < 1e-7:
    print("\n✅ Final Group Verified: All quality and acceleration metrics match.")
else:
    print("\n❌ Discrepancy detected in final group.")

--- Final Group Verification: A (2026-05-01) ---
            Feature     Manual_Value Feature_DF_Value Abs_Error
    RollingStalePct           0.0000           0.0000  0.00e+00
RollingSameVolCount           0.0000           0.0000  0.00e+00
   RollMedDollarVol 227,620,611.9000 227,620,611.9000  0.00e+00
          Convexity           0.0048           0.0048  0.00e+00

✅ Final Group Verified: All quality and acceleration metrics match.


### Group D: Complex & Relative Indicators (ATR, IR, Beta, Price Position, Slope)

In [29]:
ticker_prices = df_ohlcv.xs(verification_ticker, level="Ticker")
ticker_features = features_df.xs(verification_ticker, level="Ticker")
feat_row = ticker_features.loc[target_date]

atr_period = config.atr_period
win_63 = config.win_63d
win_20 = 20

s_idx = ticker_prices.index.get_loc(target_date)
adj_close = ticker_prices["Adj Close"]
adj_high = ticker_prices["Adj High"]
adj_low = ticker_prices["Adj Low"]
volume = ticker_prices["Volume"]


def apply_5d_slope_kernel(series, idx):
    y = series.iloc[idx - 4 : idx + 1].values
    weights = np.array([-2, -1, 0, 1, 2])
    return np.sum(y * weights) / 10


# 1. Raw ATR
prev_close = adj_close.shift(1)
tr = pd.concat(
    [(adj_high - adj_low), (adj_high - prev_close).abs(), (adj_low - prev_close).abs()],
    axis=1,
).max(axis=1)
manual_atr = tr.ewm(alpha=1 / atr_period, adjust=False).mean().loc[target_date]

# 2. Price Channel Position
h_20 = adj_high.rolling(window=win_20).max().loc[target_date]
l_20 = adj_low.rolling(window=win_20).min().loc[target_date]
manual_range_pos = (adj_close.loc[target_date] - l_20) / (h_20 - l_20)

# 3. Beta & Information Ratio (IR)
stock_rets = adj_close.pct_change()
common_idx = stock_rets.index.intersection(macro_df.index)
mkt_rets = macro_df.loc[common_idx, "Mkt_Ret"]
stock_rets_aligned = stock_rets.loc[common_idx]


s_idx_aligned = stock_rets_aligned.index.get_indexer_for(pd.Index([target_date]))[0]
y = stock_rets_aligned.iloc[s_idx_aligned - 62 : s_idx_aligned + 1]
x_mkt = mkt_rets.iloc[s_idx_aligned - 62 : s_idx_aligned + 1]

manual_beta_63 = np.cov(y, x_mkt)[0, 1] / np.var(x_mkt, ddof=1)

excess = y - x_mkt
manual_ir_63 = excess.mean() / excess.std()

# 4. Trend Slopes
manual_slope_p = apply_5d_slope_kernel(np.log(adj_close), s_idx)
v_baseline = volume.rolling(window=win_63, min_periods=1).mean().replace(0, 1e-8)
v_rel = volume / v_baseline

obv_rel = (adj_close.diff().fillna(0).pipe(np.sign) * v_rel).cumsum()
manual_slope_v = apply_5d_slope_kernel(obv_rel, s_idx)

audit_results = pd.DataFrame(
    {
        "Feature": [
            "ATR",
            "IR_63",
            "Beta_63",
            "Range_Pos_20",
            "Slope_P_5",
            "Slope_V_5",
        ],
        "Manual_Value": [
            manual_atr,
            manual_ir_63,
            manual_beta_63,
            manual_range_pos,
            manual_slope_p,
            manual_slope_v,
        ],
        "Feature_DF_Value": [
            feat_row["ATR"],
            feat_row["IR_63"],
            feat_row["Beta_63"],
            feat_row["Range_Pos_20"],
            feat_row["Slope_P_5"],
            feat_row["Slope_V_5"],
        ],
    }
)
audit_results["Abs_Error"] = (
    audit_results["Manual_Value"] - audit_results["Feature_DF_Value"]
).abs()

print(f"--- Completion Audit: {verification_ticker} ({target_date.date()}) ---")
print(
    audit_results.to_string(
        index=False,
        formatters={
            "Manual_Value": "{:,.6f}".format,
            "Feature_DF_Value": "{:,.6f}".format,
            "Abs_Error": "{:.2e}".format,
        },
    )
)

if audit_results["Abs_Error"].max() < 1e-7:
    print("\n✅ All remaining columns verified. Audit Complete.")
else:
    print("\n❌ Audit mismatch detected in remaining columns.")

--- Completion Audit: A (2026-05-01) ---
     Feature Manual_Value Feature_DF_Value Abs_Error
         ATR     3.455650         3.455650  0.00e+00
       IR_63    -0.195027        -0.195027  1.42e-15
     Beta_63     0.914108         0.914108  1.54e-14
Range_Pos_20     0.288752         0.288752  0.00e+00
   Slope_P_5    -0.001322        -0.001322  0.00e+00
   Slope_V_5    -0.285536        -0.285536  1.14e-14

✅ All remaining columns verified. Audit Complete.


# VI. Audit: Cross-Sectional Blueprints
Analyzing Z-Scores and double-scoring architectures across the universe for Pillar 5 (OBV Divergence) and Pillar 6 (Convexity).

In [30]:
target_date = pd.Timestamp("2026-04-24")
verification_ticker = "NVDA"
universe_convexity = features_df.xs(target_date, level="Date")["Convexity"]

# Manual Engine Logic
clean_universe = universe_convexity.replace([np.inf, -np.inf], np.nan).dropna()
u_mean = clean_universe.mean()
u_std = clean_universe.std()
manual_z_scores = (clean_universe - u_mean) / u_std
clip_val = config.feature_zscore_clip
final_agent_values = manual_z_scores.fillna(0).clip(-clip_val, clip_val)

raw_val = clean_universe.loc[verification_ticker]
agent_val = final_agent_values.loc[verification_ticker]

print(f"--- Pillar 6 (Convexity) Cross-Sectional Audit ---")
print(f"Date: {target_date.date()} | Universe Size: {len(clean_universe)}")
print(f"NVDA Raw Convexity    : {raw_val:.6f}")
print(f"NVDA Agent View (Z)   : {agent_val:.6f}")

--- Pillar 6 (Convexity) Cross-Sectional Audit ---
Date: 2026-04-24 | Universe Size: 1577
NVDA Raw Convexity    : 0.002689
NVDA Agent View (Z)   : 0.509011


In [31]:
from core.contracts import MetricBlueprint
from types import SimpleNamespace
from strategy.registry import get_strategy_registry

registry = get_strategy_registry(config)
daily_snapshot = features_df.xs(target_date, level="Date")
obs = SimpleNamespace(
    convexity=daily_snapshot["Convexity"],
    slope_p_5=daily_snapshot["Slope_P_5"],
    slope_v_5=daily_snapshot["Slope_V_5"],
)

convexity_blueprint = registry["Convexity"]
system_output_series = convexity_blueprint.get_agent_view(obs, config=config)
system_val = system_output_series.loc[verification_ticker]

print(f"--- Final System Verification: Pillar 6 ---")
print(f"Manual Audit Value (Z):  {agent_val:.5f}")
print(f"System Output Value (Z): {system_val:.5f}")

error = abs(agent_val - system_val)
if error < 1e-5:
    print(
        f"\n✅ VERIFIED: The System Logic matches the Manual Audit (Error: {error:.2e})"
    )
else:
    print(f"\n❌ DISCREPANCY: System output differs from manual calculation!")

--- Final System Verification: Pillar 6 ---
Manual Audit Value (Z):  0.50901
System Output Value (Z): 0.50901

✅ VERIFIED: The System Logic matches the Manual Audit (Error: 0.00e+00)


In [32]:
class QuantUtils:
    @staticmethod
    def zscore(series: pd.Series) -> pd.Series:
        clean = series.replace([np.inf, -np.inf], np.nan)
        return (clean - clean.mean()) / clean.std()


obs_p5 = SimpleNamespace(
    slope_v_5=daily_snapshot["Slope_V_5"], slope_p_5=daily_snapshot["Slope_P_5"]
)

divergence_blueprint = registry["OBV Divergence (5d)"]

# Step-by-Step Manual Double Z-Score Verification
z_vol = QuantUtils.zscore(obs_p5.slope_v_5)
z_price = QuantUtils.zscore(obs_p5.slope_p_5)
raw_divergence = z_vol - z_price

final_manual_z = QuantUtils.zscore(raw_divergence)
clip_val = config.feature_zscore_clip
manual_final_val = (
    final_manual_z.fillna(0).clip(-clip_val, clip_val).loc[verification_ticker]
)

system_output_series = divergence_blueprint.get_agent_view(obs_p5, config=config)
system_final_val = system_output_series.loc[verification_ticker]

print(f"--- Pillar 5 (OBV Divergence) Completion Audit ---")
print(f"NVDA Z-Volume (Internal): {z_vol.loc[verification_ticker]:10.5f}")
print(f"NVDA Z-Price  (Internal): {z_price.loc[verification_ticker]:10.5f}")
print(f"Manual Final Z-Score    : {manual_final_val:10.5f}")
print(f"System Final Z-Score    : {system_final_val:10.5f}")

error = abs(manual_final_val - system_final_val)
if error < 1e-5:
    print(
        f"\n✅ VERIFIED: Pillar 5 logic is mathematically consistent (Error: {error:.2e})"
    )
else:
    print(f"\n❌ DISCREPANCY: Check QuantUtils.zscore implementation details.")

--- Pillar 5 (OBV Divergence) Completion Audit ---
NVDA Z-Volume (Internal):    0.48394
NVDA Z-Price  (Internal):    0.71172
Manual Final Z-Score    :   -0.31402
System Final Z-Score    :   -0.31402

✅ VERIFIED: Pillar 5 logic is mathematically consistent (Error: 0.00e+00)


# VII. Data Exports & Sheets

In [33]:
filename = "verify_UI_n_features_calc.xlsx"
SU.export_audit_to_excel(
    audit_pack=analyzer1.last_run,
    filename=filename,
)
print(f"\n{filename} written to {OUTPUT_DIR}\\")

[FILE] [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\verify_UI_n_features_calc.xlsx
[DONE] Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\verify_UI_n_features_calc.xlsx

verify_UI_n_features_calc.xlsx written to C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\


In [34]:
features_df.loc["TSLA"].to_excel("features_df_out.xlsx")
print("Wrote TSLA data from features_df to features_df_out.xlsx")

Wrote TSLA data from features_df to features_df_out.xlsx


In [35]:
_nb_elapsed = time.perf_counter() - _nb_start_time
print(f"\n⏱️ Total Execution Time: {_nb_elapsed:.2f}s ({_nb_elapsed/60:.2f} min)")


⏱️ Total Execution Time: 195.49s (3.26 min)
